<a href="https://colab.research.google.com/github/Atena-Rashidi/Advanced_GenAI_Techniques/blob/main/Integration_of_LLM_with_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integration of Large Language Models (LLMs) with Retrieval-Augmented Generation (RAG) from Scratch
Integrating Large Language Models (LLMs) with Retrieval-Augmented Generation (RAG) enhances LLMs by incorporating external knowledge sources, improving their ability to generate factually accurate and context-aware responses. Below is a detailed, step-by-step guide to implementing LLM + RAG from scratch.


## 1. Understanding RAG

**1.1. What is RAG?**

RAG is a framework where an LLM retrieves relevant documents from an external knowledge base and generates responses based on them, rather than relying solely on its internal knowledge. This is useful for:
- Reducing hallucinations
- Providing up-to-date information
- Integrating specialized domain knowledge

**1.2. Architecture of RAG**

A RAG system consists of two main components:
1. **Retriever** – Fetches relevant documents from a knowledge base.
2. **Generator** – Uses an LLM to generate responses based on retrieved documents.


### 1. Retriever
The **Retriever** is responsible for fetching relevant documents or pieces of information from a large knowledge base or dataset. Here's a more detailed look at its role:

- **Function**: The retriever searches through a vast amount of data to find the most relevant documents or snippets that match a given query or context.
- **Techniques**: It often uses techniques like keyword matching, semantic search, or more advanced methods like dense retrieval using embeddings.
- **Efficiency**: The goal is to quickly and accurately retrieve information that is most likely to be useful for generating a response.
- **Example**: If you ask a question about machine learning, the retriever might fetch articles, papers, or sections of books that discuss machine learning concepts.

### 2. Generator
The **Generator** uses a large language model (LLM) to create coherent and contextually appropriate responses based on the documents retrieved by the retriever. Here's a closer look at its role:

- **Function**: The generator takes the retrieved documents and uses them to generate a response that is informative and relevant to the user's query.
- **Techniques**: It leverages the capabilities of LLMs, such as GPT-4, to understand the context and generate human-like text.
- **Contextualization**: The generator ensures that the response is not just a regurgitation of the retrieved documents but is synthesized and contextualized to address the specific query.
- **Example**: Continuing with the machine learning example, the generator would use the retrieved documents to craft a detailed explanation or answer to your question about machine learning.

### How They Work Together
- **Integration**: The retriever and generator work in tandem. The retriever first narrows down the vast amount of information to a manageable set of relevant documents. The generator then uses this focused set of information to produce a high-quality response.
- **Feedback Loop**: In some advanced systems, there might be a feedback loop where the generator's output can influence further retrieval, refining the process iteratively.


## 2. Step-by-Step Implementation of LLM + RAG

We will integrate an LLM (e.g., OpenAI GPT-4, Llama 2, or Mistral) with a vector database (e.g., FAISS, Pinecone, Weaviate) for retrieval.

**Step 1. Set Up the Environment**

You need Python and essential libraries installed. Install the necessary dependencies:

In [ ]:
!pip install langchain openai faiss-cpu sentence-transformers chromadb
!pip install -U langchain-community
!pip install langchain-ollama

**Step 2: Load and Embed Documents**

First, prepare a set of documents to use as the knowledge base.

**2.1 Load Documents**

You can load text files, PDFs, or web pages.

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load documents
loader = TextLoader("knowledge_base.txt")  # Your knowledge base file
documents = loader.load()

**2.2. Split Text into Smaller Chunks**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

**Step 3: Convert Documents into Embeddings**

To enable retrieval, we convert text chunks into embeddings using a Sentence Transformer.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert document chunks into embeddings
doc_embeddings = np.array(embedding_model.encode([doc.page_content for doc in docs],
                                                 normalize_embeddings=True))

**Step 4: Store Embeddings in a Vector Database**

We use FAISS to store and retrieve embeddings efficiently.

Given a user query, convert it into an embedding and retrieve the closest document chunks using cosine similarity.

### How Retrieval Works

1. Split documents into smaller chunks.
2. Convert each chunk into a vector embedding.
3. Convert the user’s query into an embedding.
4. Use cosine similarity (or L2 distance) to find the most relevant chunks.

### Cosine Similarity Formula

For two vectors \( A \) and \( B \), cosine similarity is:

$$
\text{cosine_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}
$$

where:

- \( $A \cdot B $\) is the dot product of the two vectors.
- \( \|A\| \) and \( \|B\| \) are the L2 norms (magnitudes) of the vectors.

Values range from -1 (completely opposite) to 1 (exact match).


In [ ]:
import faiss
# Use Inner Product for cosine similarity (normalize vectors)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

### Step 5: Retrieve Top-k Relevant Chunks

After calculating the cosine similarity between the user query embedding and each document chunk embedding, the next step is to retrieve the top-k most relevant chunks. Here's how it works:

1. **Sort Similarities**: Arrange the document chunks based on their cosine similarity scores in descending order.
2. **Select Top-k**: Choose the top-k chunks with the highest similarity scores. The value of k can be adjusted based on how many relevant chunks you want to retrieve.
3. **Return Chunks**: These top-k chunks are then returned as the most relevant pieces of information for the user's query.

This step ensures that you get the most pertinent information from the document, making the retrieval process efficient and effective.

In [ ]:
def retrieve_top_k(query, k=3):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(np.array(query_embedding), k)
    retrieved_docs = [docs[i].page_content for i in indices[0]]
    return retrieved_docs

# Example
query = "What is Retrieval-Augmented Generation?"
retrieved_docs = retrieve_top_k(query)

for doc in retrieved_docs:
    print(doc)


1. Improved Accuracy
Contextual Information: By retrieving relevant documents, the RAG system provides the LLM with additional context, leading to more accurate and informed responses.
Reduced Hallucinations: The LLM is less likely to generate incorrect or fabricated information because it relies on factual documents.
2. Efficiency
Focused Retrieval: The retriever narrows down the vast amount of data to the most relevant chunks, making the generation process more efficient.
Resource Optimization: By working with smaller, relevant chunks of data, the system uses computational resources more effectively.
3. Enhanced Relevance
Tailored Responses: The augmentation step ensures that the information provided is directly relevant to the user's query, leading to more meaningful and useful responses.
Dynamic Adaptation: The system can adapt to different queries by retrieving and augmenting different sets of documents, making it versatile.
4. Scalability
Trust and Reliability: Users are more lik

### Step 6: Integrate with an LLM for Response Generation

Now, we use a Large Language Model (LLM), such as OpenAI's GPT-4 or a local model like LLaMA 2, to generate responses based on the retrieved documents. Here's how it works:

1. **Input Preparation**: Combine the top-k relevant document chunks into a single input for the LLM. This input should provide enough context for the model to generate a coherent and relevant response.
2. **Response Generation**: Feed the prepared input into the LLM. The model will process the information and generate a response that addresses the user's query.
3. **Post-Processing**: Optionally, refine the generated response to ensure clarity, coherence, and relevance. This might involve minor edits or rephrasing.


### Initialize OpenAI model

In [ ]:
from langchain.chat_models import ChatOpenAI

# Initialize OpenAI model (replace with your API key)
llm = ChatOpenAI(model_name="gpt-4", openai_api_key="HF_TOKEN")

def generate_response(query):
    retrieved_docs = retrieve_top_k(query)
    context = "\n".join(retrieved_docs)

    prompt = f"Use the following context to answer the question:\n\n{context}\n\nQuestion: {query}"

    response = llm.predict(prompt)
    return response

# Example Query
query = "How does RAG improve LLM performance?"
print(generate_response(query))

 ### Initialize LLaMA model

In [ ]:
from langchain_ollama import ChatOllama

# Initialize LLaMA model (replace with your API key)
llm = ChatOllama(model="llama-2", api_key="HF_TOKEN")

def generate_response(query):
    retrieved_docs = retrieve_top_k(query)
    context = "\n".join(retrieved_docs)

    prompt = f"Use the following context to answer the question:\n\n{context}\n\nQuestion: {query}"

    response = llm.predict(prompt)
    return response

# Example Query
query = "How does RAG improve LLM performance?"
print(generate_response(query))

**7. Optimizations for Better RAG Performan**ce
1. **Use a more advanced embedding model**
   - Upgrade to all-mpnet-base-v2 or domain-specific embeddings.
2. **Use a better retrieval strategy**
   - Hybrid search: Combine keyword search + vector search for improved retrieval.
3. **Fine-tune LLM for better responses**
   - Fine-tune an open-source model (e.g., LLaMA 2, Mistral) with domain-specific data.
4. **Use memory for better context retention**
   - Implement LangChain’s Conversational Memory to store past interactions.

**8. Key Takeaways**

- Yes, we retrieve document chunks before similarity computation.
- Cosine similarity is commonly used for text retrieval.
- FAISS uses L2 distance by default, but by normalizing vectors, it effectively works as cosine similarity.
- Hybrid retrieval (combining keyword search + vector similarity) can improve accuracy.